# 24 - Aplicacion backend sencilla con pandas y Titanic (Kaggle)

## Objetivos de aprendizaje

En esta sesion aprenderas a:

1. Entender los pasos minimos para construir una aplicacion backend guiada por datos.
2. Cargar y validar un dataset real de Kaggle con `pandas`.
3. Hacer exploracion inicial para detectar estructura, tipos y faltantes.
4. Realizar una limpieza sencilla y defendible para un flujo productivo basico.
5. Escribir consultas con `query` y filtros booleanos de `pandas`.
6. Construir reportes con `groupby` y agregaciones.
7. Encapsular logica en funciones tipo servicio backend.
8. Evitar errores comunes que rompen analisis o endpoints.


## Ruta de la sesion

1. Que significa backend orientado a datos.
2. Caso de uso y dataset (Titanic de Kaggle).
3. Carga robusta y validacion de columnas esperadas.
4. Exploracion inicial con `pandas`.
5. Limpieza sencilla y creacion de variables utiles.
6. Consultas con `query`.
7. Consultas con filtros booleanos y `loc`.
8. Reportes de negocio con `groupby`.
9. Mini backend: funciones reutilizables de consulta y reporte.
10. Errores comunes + ejercicios de pensamiento.


## 1) Que es una aplicacion backend orientada a datos

Una aplicacion backend recibe peticiones, ejecuta reglas y devuelve respuestas.
Cuando el backend es orientado a datos, su valor esta en transformar tablas crudas en informacion accionable.

Flujo minimo:

1. **Entrada**: leer datos de una fuente (CSV, BD, API).
2. **Validacion**: confirmar esquema y calidad minima.
3. **Transformacion**: limpiar, tipar y enriquecer.
4. **Consulta**: filtrar y responder preguntas puntuales.
5. **Agregacion**: resumir para apoyar decisiones.
6. **Salida**: entregar estructuras listas para un endpoint (dict, JSON o DataFrame).

En esta clase no construiremos un servidor HTTP completo; construiremos la logica central que un backend real reutiliza.


## 2) Caso de uso: Titanic (Kaggle)

Usaremos el dataset clasico de Kaggle:

- Competencia: **Titanic - Machine Learning from Disaster**.
- Archivo recomendado: `train.csv`.

Preguntas de negocio que simularemos:

1. Que perfil de pasajeros tuvo mayor supervivencia.
2. Como consultar segmentos especificos (edad, clase, puerto, tarifa).
3. Como generar un resumen que un endpoint backend pueda devolver.

Coloca el archivo descargado como uno de estos nombres:

- `data/raw/train.csv`
- `data/raw/titanic_train.csv`
- `data/raw/titanic.csv`


In [ ]:
from pathlib import Path

import pandas as pd


In [ ]:
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


## 3) Carga robusta y validacion minima

Buenas practicas para backend:

1. No asumir rutas fijas sin fallback.
2. Verificar columnas obligatorias.
3. Fallar rapido con mensajes claros cuando falta informacion.


In [ ]:
EXPECTED_COLUMNS = {
    "PassengerId",
    "Survived",
    "Pclass",
    "Name",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Ticket",
    "Fare",
    "Cabin",
    "Embarked",
}


def cargar_titanic() -> tuple[pd.DataFrame, Path]:
    candidatos = [
        Path("data/raw/train.csv"),
        Path("data/raw/titanic_train.csv"),
        Path("data/raw/titanic.csv"),
    ]

    for ruta in candidatos:
        if ruta.exists():
            df_local = pd.read_csv(ruta)
            faltantes = EXPECTED_COLUMNS.difference(df_local.columns)
            if faltantes:
                raise ValueError(
                    "El archivo existe pero no trae columnas clave: "
                    f"{sorted(faltantes)}"
                )
            return df_local, ruta

    raise FileNotFoundError(
        "No se encontro Titanic en data/raw/. Descarga train.csv desde Kaggle y colocalo en data/raw/."
    )


In [ ]:
df_raw, data_path = cargar_titanic()

print("Dataset cargado desde:", data_path)
print("Forma inicial:", df_raw.shape)
df_raw.head(3)


## 4) Exploracion inicial (EDA rapida)

Antes de limpiar o consultar, responde:

1. Cuantas filas y columnas hay.
2. Que tipos tiene cada variable.
3. Donde hay faltantes y de que magnitud.
4. Que rangos y distribuciones parecen razonables.

Objetivo: entender la geometria de los datos para no limpiar a ciegas.


In [ ]:
print("Forma:", df_raw.shape)
print("\nTipos de columnas:")
print(df_raw.dtypes)


In [ ]:
faltantes = df_raw.isna().sum().sort_values(ascending=False)
faltantes = faltantes[faltantes > 0]

print("Columnas con valores faltantes:")
print(faltantes)


In [ ]:
print("Resumen numerico:")
print(df_raw.describe(numeric_only=True).T)

print("\nResumen categorico:")
print(df_raw.describe(include=["object"]).T)


## 5) Limpieza sencilla y trazable

Limpieza no significa borrar todo lo incomodo.
Significa aplicar reglas justificadas por el problema.

Estrategia de esta clase:

1. Estandarizar texto para evitar categorias duplicadas por formato.
2. Eliminar duplicados de identificador si aparecen.
3. Imputar `Age` por mediana de grupo (`Pclass`, `Sex`) y fallback global.
4. Imputar `Fare` por mediana de `Pclass`.
5. Completar `Embarked` con moda global.
6. Crear variables derivadas utiles para consulta (`family_size`, `is_alone`).
7. Ajustar tipos para usar memoria de forma mas eficiente.


In [ ]:
def limpiar_titanic(df: pd.DataFrame) -> pd.DataFrame:
    df_clean = df.copy()

    columnas_texto = ["Name", "Sex", "Ticket", "Cabin", "Embarked"]
    for col in columnas_texto:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].astype("string").str.strip()

    if "PassengerId" in df_clean.columns:
        df_clean = df_clean.drop_duplicates(subset=["PassengerId"], keep="first")
    else:
        df_clean = df_clean.drop_duplicates()

    df_clean["Sex"] = df_clean["Sex"].str.lower()
    df_clean["Embarked"] = df_clean["Embarked"].replace({"": pd.NA}).str.upper()

    edad_por_grupo = df_clean.groupby(["Pclass", "Sex"], dropna=False)["Age"].transform("median")
    df_clean["Age"] = df_clean["Age"].fillna(edad_por_grupo)
    df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].median())

    tarifa_por_clase = df_clean.groupby("Pclass", dropna=False)["Fare"].transform("median")
    df_clean["Fare"] = df_clean["Fare"].fillna(tarifa_por_clase)
    df_clean["Fare"] = df_clean["Fare"].fillna(df_clean["Fare"].median())

    modo_embarked = df_clean["Embarked"].mode(dropna=True)
    if not modo_embarked.empty:
        df_clean["Embarked"] = df_clean["Embarked"].fillna(modo_embarked.iloc[0])

    df_clean["Cabin"] = df_clean["Cabin"].fillna("")
    df_clean["has_cabin_info"] = df_clean["Cabin"].ne("").astype("int8")
    df_clean["family_size"] = df_clean["SibSp"] + df_clean["Parch"] + 1
    df_clean["is_alone"] = (df_clean["family_size"] == 1).astype("int8")

    df_clean["Survived"] = df_clean["Survived"].astype("int8")
    df_clean["Pclass"] = df_clean["Pclass"].astype("int8")
    df_clean["Sex"] = df_clean["Sex"].astype("category")
    df_clean["Embarked"] = df_clean["Embarked"].astype("category")

    return df_clean


In [ ]:
df = limpiar_titanic(df_raw)

print("Forma original:", df_raw.shape)
print("Forma limpia:", df.shape)

print("\nTop columnas con faltantes despues de limpiar:")
print(df.isna().sum().sort_values(ascending=False).head(10))

df.head(3)


## 6) Consultas con `query`

`query` es util cuando quieres escribir filtros como expresiones legibles.
Ventajas:

1. Sintaxis compacta para reglas de negocio.
2. Facil de convertir en filtros dinamicos.
3. Buena legibilidad cuando se comparte codigo en equipo.

Regla clave: para variables de Python dentro de `query`, usa `@variable`.


In [ ]:
consulta_1 = df.query("Age >= 18 and Pclass == 1 and Fare >= 50")[
    ["PassengerId", "Name", "Sex", "Age", "Pclass", "Fare", "Survived"]
]

print("Adultos en primera clase con tarifa >= 50")
consulta_1.head(10)


In [ ]:
puertos_objetivo = ["C", "Q"]
edad_minima = 30

consulta_2 = df.query(
    "Sex == 'female' and Embarked in @puertos_objetivo and Age >= @edad_minima"
)[["PassengerId", "Name", "Embarked", "Age", "Fare", "Survived"]]

print("Mujeres de puertos C/Q con edad minima 30")
consulta_2.head(10)


## 7) Filtros booleanos con `loc`

Cuando la condicion es mas compleja o requiere funciones de serie, `loc` suele ser mas claro que `query`.
Patron recomendado:

1. Construir una mascara booleana con parentesis claros.
2. Aplicarla con `df.loc[mascara, columnas]`.
3. Ordenar y limitar para devolver respuestas utiles.


In [ ]:
filtro_hombres = (
    (df["Sex"] == "male")
    & (df["Age"].between(20, 40))
    & (df["SibSp"] >= 1)
    & (df["Survived"] == 0)
)

hombres_jovenes_no_supervivientes = df.loc[
    filtro_hombres,
    ["PassengerId", "Name", "Age", "Pclass", "SibSp", "Parch", "Fare", "Embarked"],
].sort_values("Fare", ascending=False)

hombres_jovenes_no_supervivientes.head(10)


In [ ]:
def filtrar_pasajeros(
    df: pd.DataFrame,
    *,
    sexo: str | None = None,
    edad_min: float | None = None,
    edad_max: float | None = None,
    clase: int | None = None,
    sobrevivio: int | None = None,
    limite: int = 20,
) -> pd.DataFrame:
    mascara = pd.Series(True, index=df.index)

    if sexo is not None:
        mascara &= df["Sex"] == sexo
    if edad_min is not None:
        mascara &= df["Age"] >= edad_min
    if edad_max is not None:
        mascara &= df["Age"] <= edad_max
    if clase is not None:
        mascara &= df["Pclass"] == clase
    if sobrevivio is not None:
        mascara &= df["Survived"] == sobrevivio

    columnas_salida = ["PassengerId", "Name", "Sex", "Age", "Pclass", "Fare", "Embarked", "Survived"]
    return (
        df.loc[mascara, columnas_salida]
        .sort_values(["Pclass", "Fare"], ascending=[True, False])
        .head(limite)
    )


In [ ]:
filtrar_pasajeros(df, sexo="female", edad_min=18, clase=1, sobrevivio=1, limite=10)


## 8) Reportes con `groupby`

`groupby` es la base de un backend analitico: convierte miles de filas en indicadores.
Concepto:

1. Dividir los datos por categoria(s).
2. Aplicar funciones de agregacion (`count`, `mean`, `sum`, etc.).
3. Interpretar el resultado para tomar decisiones.


In [ ]:
supervivencia_por_sexo = (
    df.groupby("Sex", observed=True)["Survived"]
    .agg(["count", "mean"])
    .rename(columns={"count": "pasajeros", "mean": "tasa_supervivencia"})
    .sort_values("tasa_supervivencia", ascending=False)
)

supervivencia_por_sexo


In [ ]:
reporte_embarque_clase = (
    df.groupby(["Embarked", "Pclass"], observed=True)
    .agg(
        pasajeros=("PassengerId", "count"),
        tasa_supervivencia=("Survived", "mean"),
        edad_media=("Age", "mean"),
        tarifa_media=("Fare", "mean"),
    )
    .reset_index()
    .sort_values(["Embarked", "Pclass"])
)

reporte_embarque_clase


In [ ]:
df["fare_promedio_por_clase"] = df.groupby("Pclass", dropna=False)["Fare"].transform("mean")
df["fare_relativo_vs_clase"] = df["Fare"] / df["fare_promedio_por_clase"]

df[["PassengerId", "Pclass", "Fare", "fare_promedio_por_clase", "fare_relativo_vs_clase"]].head(10)


## 9) Mini aplicacion backend: capa de servicio con pandas

Patron recomendado para proyectos universitarios:

1. Una funcion para limpiar/cargar datos.
2. Funciones puras de consulta (sin I/O) para endpoints.
3. Funciones de reporte agregadas para dashboards.

Esto facilita pruebas y reduce errores cuando despues agregas FastAPI o Flask.


In [ ]:
def resumen_general(df: pd.DataFrame) -> dict:
    return {
        "total_pasajeros": int(len(df)),
        "tasa_supervivencia": float(df["Survived"].mean()),
        "edad_promedio": float(df["Age"].mean()),
        "tarifa_promedio": float(df["Fare"].mean()),
    }


def reporte_supervivencia(df: pd.DataFrame, columnas: list[str]) -> pd.DataFrame:
    return (
        df.groupby(columnas, observed=True)["Survived"]
        .agg(pasajeros="count", tasa_supervivencia="mean")
        .reset_index()
        .sort_values("tasa_supervivencia", ascending=False)
    )


In [ ]:
def endpoint_health() -> dict:
    return {"status": "ok"}


def endpoint_resumen(df: pd.DataFrame) -> dict:
    return {
        "status": "ok",
        "data": resumen_general(df),
    }


def endpoint_consulta_demo(df: pd.DataFrame) -> dict:
    data = filtrar_pasajeros(df, sexo="female", edad_min=18, clase=1, sobrevivio=1, limite=5)
    return {
        "status": "ok",
        "rows": len(data),
        "data": data.to_dict(orient="records"),
    }


In [ ]:
print("Health:", endpoint_health())
print("\nResumen backend:")
print(endpoint_resumen(df))

print("\nConsulta backend de ejemplo (primeras filas):")
payload_demo = endpoint_consulta_demo(df)
print({k: payload_demo[k] for k in ["status", "rows"]})
pd.DataFrame(payload_demo["data"]).head(5)


## 10) Errores comunes en pandas (y como evitarlos)

1. **`SettingWithCopyWarning`** por encadenar indexado.
   - Mal: `df[df['Age'] > 30]['Fare'] = 0`
   - Bien: usar `loc` con mascara.

2. **Olvidar `@` en `query`** cuando usas variables externas.
   - Mal: `df.query('Fare > umbral')`
   - Bien: `df.query('Fare > @umbral')`

3. **Comparar texto sin normalizar** (`Male`, `male`, ` male `).
   - Solucion: estandarizar con `.str.strip().str.lower()` antes de filtrar.

4. **Imputar todo con una sola media global sin pensar en contexto**.
   - Solucion: imputar por grupos cuando tenga sentido (ej. `Pclass`, `Sex`).

5. **Perder trazabilidad de cambios**.
   - Solucion: encapsular limpieza en funciones y documentar reglas.

6. **No validar columnas al cargar**.
   - Solucion: schema check temprano y errores explicitos.


In [ ]:
# Ejemplo correcto para evitar SettingWithCopyWarning
df_demo = df.copy()
mascara_edad = df_demo["Age"] > 30
df_demo.loc[mascara_edad, "segmento_edad"] = "mayor_30"
df_demo.loc[~mascara_edad, "segmento_edad"] = "menor_igual_30"

df_demo[["PassengerId", "Age", "segmento_edad"]].head(10)


In [ ]:
# Ejemplo correcto de variable externa en query
umbral_tarifa = 80
df.query("Fare > @umbral_tarifa")[["PassengerId", "Name", "Fare", "Pclass"]].head(10)


## 11) Ejercicios de pensamiento (no copiar/pegar)

Meta: practicar criterio tecnico. Cada ejercicio pide justificar decisiones, no solo producir output.


### Ejercicio 1: diseña el contrato de un endpoint real

Diseña el endpoint `GET /pasajeros` para esta aplicacion.

Define:

1. Parametros admitidos (ej. `sexo`, `edad_min`, `clase`, `sobrevivio`).
2. Reglas de validacion (tipos, rangos, defaults).
3. Estructura de respuesta JSON para exito y para error.
4. Como paginarias resultados (limite y offset).


In [ ]:
# Escribe aqui tu contrato de endpoint y dos ejemplos de request/response.


### Ejercicio 2: limpieza con impacto de negocio

Propone una estrategia alternativa para imputar `Age`.

Compara contra la estrategia de clase y responde:

1. Que supuesto estadistico estas haciendo.
2. Que sesgo podrias introducir.
3. En que tipo de decision backend te podria afectar.


In [ ]:
# Implementa y compara dos estrategias de imputacion de Age.


### Ejercicio 3: `query` vs filtros booleanos

Resuelve la misma pregunta con ambos enfoques:

- "Pasajeros de segunda o tercera clase, con Fare < 20, que sobrevivieron".

Luego explica:

1. Cual version te parece mas mantenible y por que.
2. Que riesgos de bug ves en cada estilo.
3. En que caso usarias uno u otro dentro de una API.


In [ ]:
# Escribe aqui ambas versiones y compara resultados.


### Ejercicio 4: leer `groupby` como si fueras PM o analista

Construye un reporte por `Sex`, `Pclass` y `Embarked` con:

1. total de pasajeros,
2. tasa de supervivencia,
3. tarifa promedio.

Despues redacta 3 conclusiones y 2 advertencias metodologicas.
No uses lenguaje tecnico excesivo: imagina que se lo explicas a un equipo no tecnico.


In [ ]:
# Construye el reporte y redacta tus hallazgos ejecutivos.


### Ejercicio 5: llevar esto a una API con FastAPI

Sin implementar aun el servidor completo, define un plan tecnico:

1. Estructura de carpetas del proyecto.
2. Donde viviria la logica de `pandas` (servicios).
3. Donde vivirian los esquemas de entrada/salida.
4. Como escribirias al menos 3 pruebas unitarias clave.
5. Que metricas monitorearias en produccion.


In [ ]:
# Escribe aqui tu plan tecnico de migracion a API.


## 12) Cierre

Ideas clave:

1. Un backend de datos empieza por validar y limpiar antes de responder consultas.
2. `query` y filtros booleanos son complementarios; elige segun legibilidad y complejidad.
3. `groupby` traduce filas en decisiones de negocio.
4. Encapsular logica en funciones facilita pruebas y despliegue posterior.
5. La calidad de una API analitica depende mas del criterio de datos que del framework web.
